# Historical Transport Resilience Index (TRI) — Proof of Concept

**Type:** Historical / retrospective geospatial resilience assessment
**Scope:** This notebook computes the TRI from a single historical flood event using
observed-style (here: synthetically generated) road, flood, facility, and travel-survey data.

## What this notebook does

| Stage | Output |
|---|---|
| 1. Infrastructure Functionality (IF) | Road serviceability under a historical flood, by ward |
| 2. Accessibility Index (AI) | Share of essential facilities reachable through the surviving network, by ward |
| 3. Behavioural Adaptation (BA) | How travellers actually responded (delay, mode change, cancellations, rerouting), by ward |
| 4. Transport Resilience Index (TRI) | `TRI = (IF + AI + BA) / 3`, by ward |

## Explicitly out of scope

This is a **historical assessment tool only**. It does **not** contain, and will never contain:
- Machine Learning / Deep Learning
- Predictive or forecasting models
- Future / "what-if" scenario simulation
- LLMs, RAG, or any generative-AI component

Every number produced here describes **what already happened**, computed with transparent,
auditable, rule-based formulas.

## How the notebook is organised

The notebook is split into four parts (this is **Part 1**):

- **Part 1** — Project setup, package installation, folder structure, dummy data generation, helper utilities *(this notebook)*
- **Part 2** — Road–Flood Overlay → Road Status → Infrastructure Functionality (IF) → Accessibility Index (AI)
- **Part 3** — Travel Behaviour → Behavioural Adaptation (BA) → Master Dataset → Transport Resilience Index (TRI)
- **Part 4** — Visualization, validation reporting, and `run_pipeline()` end-to-end execution

Every intermediate result is written to disk as a shapefile / GeoTIFF / CSV, so each function in
later parts can be re-run independently by reading the files produced by the previous step.


## System Architecture

```
Road Network  ─┐
               ├──► Road–Flood Overlay ──► Road Status ──► Infrastructure Functionality (IF)
Flood Raster  ─┘                                                        │
                                                                         ▼
Essential Facilities ───────────────────────────────────────►  Accessibility Index (AI)
                                                                         │
Travel Survey ──► Travel Behaviour Metrics ──► Behavioural Adaptation (BA)
                                                                         │
                                                                         ▼
                                              IF + AI + BA ──► Master Dataset ──► TRI
```

**Reading order:** Road geometry is overlaid on the flood raster to get an average flood depth
per road segment → each segment is classified Operational / Partial / Closed → the *effective
length* of the network (weighted by that classification) per ward gives **IF**. The road-status
network (with Closed roads removed) is used as a graph to test which facilities each ward can
still reach, giving **AI**. Independently, the travel survey tells us what travellers actually
experienced (delay, cancellations, mode/route changes), aggregated per ward into **BA**. The
three ward-level indices are combined into the final **TRI**.


## Project folder structure

Everything is created automatically under `/content/project` (Colab) — nothing needs to be
uploaded manually.

```
project/
├── input/
│   ├── roads/          road_network.shp
│   ├── flood/           flood_depth.tif
│   ├── facilities/      facilities.shp
│   ├── travel/          travel_survey.csv
│   └── boundary/        boundary.shp
├── intermediate/
│   ├── 01_overlay/      road segments + avg_flood_depth
│   ├── 02_status/       road segments + status classification
│   ├── 03_if/           if.csv
│   ├── 04_accessibility/ accessibility.csv
│   ├── 05_behaviour/    behaviour.csv
│   ├── 06_ba/           ba.csv
│   ├── 07_master/       master_resilience.csv
│   └── 08_tri/          tri.csv
├── output/
│   ├── maps/            PNG visualizations
│   ├── csv/             final CSV exports
│   └── shapefiles/      final shapefile exports
├── dummy_data/          copies of raw synthetic inputs (for inspection)
└── logs/                pipeline run logs
```


## Data Dictionary

### 1. Road Network (`input/roads/roads.shp`) — LineString

| Field | Type | Description | Units | Example |
|---|---|---|---|---|
| road_id | string | Unique road segment identifier | — | R001 |
| road_type | string | Functional road class | — | primary / secondary / residential |
| length_m | float | Segment length | metres | 1500.0 |
| speed_lim* | int | Posted speed limit | km/h | 60 |
| lanes | int | Number of lanes | count | 2 |
| status | string | Filled in later (Operational/Partial/Closed) | — | Operational |

\* stored as `speed_lim` in the shapefile because ESRI Shapefile field names are limited to 10
characters; conceptually this is `speed_limit`.

### 2. Flood Raster (`input/flood/flood_depth.tif`) — GeoTIFF, Float32, 10 m resolution

| Attribute | Description | Units |
|---|---|---|
| Pixel value | Flood inundation depth during the historical event | metres |
| Resolution | 10 × 10 | metres/pixel |
| nodata | -9999 | — |

### 3. Administrative Boundary (`input/boundary/boundary.shp`) — Polygon

| Field | Type | Description | Example |
|---|---|---|---|
| ward_id | string | Unique ward identifier | W01 |
| ward_name | string | Ward display name | Ward 1 |

### 4. Essential Facilities (`input/facilities/facilities.shp`) — Point

| Field | Type | Description | Example |
|---|---|---|---|
| fac_id* | string | Unique facility identifier | F001 |
| fac_type* | string | Facility category | Hospital / School / Market / Bus Stop / Metro Station |

\* stored as `fac_id` / `fac_type` (10-character shapefile field limit); conceptually
`facility_id` / `facility_type`.

### 5. Travel Survey (`input/travel/travel_survey.csv`) — CSV

| Field | Type | Description | Units | Example |
|---|---|---|---|---|
| trip_id | string | Unique trip identifier | — | T0001 |
| person_id | string | Unique respondent identifier | — | P0042 |
| origin_x, origin_y | float | Trip origin coordinates | metres (projected CRS) | 1234.5, 987.6 |
| destination_x, destination_y | float | Trip destination coordinates | metres (projected CRS) | 2200.0, 1500.0 |
| planned_mode | string | Mode the traveller intended to use | — | car |
| actual_mode | string | Mode actually used | — | bus |
| travel_time_before | float | Typical (pre-event) travel time | minutes | 25.0 |
| travel_time_after | float | Observed travel time during/after the event | minutes | 48.0 |
| trip_cancelled | bool | Whether the trip was abandoned | — | False |
| route_changed | bool | Whether the traveller rerouted | — | True |
| ward_id | string | Ward the trip originates in | — | W03 |

All coordinates use the same projected CRS as the spatial layers (**EPSG:32646**) so that roads,
boundary, facilities, flood raster, and travel origins/destinations are all spatially consistent
and directly overlay one another.


## Cell: Install required packages

Colab ships with some geospatial packages but not all of them at compatible versions, so we pin
installs explicitly. This cell is safe to re-run.

In [ ]:
# Install required packages (Colab-safe, idempotent)
!pip install -q geopandas rasterio rasterstats networkx shapely scipy matplotlib
print("Packages installed.")

## Cell: Import libraries

In [ ]:
from __future__ import annotations

import os
import logging
import warnings
import math
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
import rasterstats
import networkx as nx
from shapely.geometry import LineString, Point, Polygon, box
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Libraries imported successfully.")

## Cell: Create project folder structure

In [ ]:
# ---------------------------------------------------------------------------
# Function: create_project_structure
# Purpose : Create the full directory tree used by the entire pipeline.
# Inputs  : base_dir (str) - root folder for the project
# Output  : Nested folders on disk (no files)
# Returns : dict[str, Path] mapping a short key to every created folder
# ---------------------------------------------------------------------------
def create_project_structure(base_dir: str = "/content/project") -> dict:
    """Create the standard TRI project folder structure.

    Parameters
    ----------
    base_dir : str
        Root directory under which the project tree is created.

    Returns
    -------
    dict[str, Path]
        Mapping of short keys (e.g. "roads", "if", "maps") to their
        absolute Path objects, for convenient reuse in later cells.
    """
    base = Path(base_dir)
    subfolders = {
        "roads": "input/roads",
        "flood": "input/flood",
        "facilities": "input/facilities",
        "travel": "input/travel",
        "boundary": "input/boundary",
        "overlay": "intermediate/01_overlay",
        "status": "intermediate/02_status",
        "if": "intermediate/03_if",
        "accessibility": "intermediate/04_accessibility",
        "behaviour": "intermediate/05_behaviour",
        "ba": "intermediate/06_ba",
        "master": "intermediate/07_master",
        "tri": "intermediate/08_tri",
        "maps": "output/maps",
        "csv": "output/csv",
        "shapefiles": "output/shapefiles",
        "dummy_data": "dummy_data",
        "logs": "logs",
    }
    paths = {}
    for key, rel in subfolders.items():
        p = base / rel
        p.mkdir(parents=True, exist_ok=True)
        paths[key] = p
    paths["base"] = base
    return paths


PROJECT = create_project_structure("/content/project")

# ---------------------------------------------------------------------------
# Logging setup: every function in this pipeline logs to logs/pipeline.log
# ---------------------------------------------------------------------------
LOG_PATH = PROJECT["logs"] / "pipeline.log"
logger = logging.getLogger("TRI")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fh = logging.FileHandler(LOG_PATH)
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(sh)

logger.info("Project structure created at %s", PROJECT["base"])
for k, v in PROJECT.items():
    print(f"{k:12s} -> {v}")

## Cell: Validation helper

A single reusable helper is used after **every** function in this pipeline (per the project
specification) to print input/output row counts, missing values, summary statistics, and the
file location that was written.

In [ ]:
# ---------------------------------------------------------------------------
# Function: validate_output
# Purpose : Standard post-processing validation report used after every
#           pipeline function (vector, raster, or tabular output).
# Inputs  : label (str), df_in (optional DataFrame/GeoDataFrame),
#           df_out (optional DataFrame/GeoDataFrame), file_path (str)
# Output  : Printed report (no file written)
# Returns : None
# ---------------------------------------------------------------------------
def validate_output(label: str, df_out, file_path: str, df_in=None) -> None:
    """Print a standard validation report for a pipeline step.

    Parameters
    ----------
    label : str
        Human-readable name of the processing step (e.g. "Road-Flood Overlay").
    df_out : DataFrame or GeoDataFrame
        The output dataset produced by the step.
    file_path : str
        Path to the file that was written to disk.
    df_in : DataFrame or GeoDataFrame, optional
        The input dataset, if row-count comparison is meaningful.
    """
    print(f"\n--- Validation: {label} ---")
    if df_in is not None:
        print(f"Input rows : {len(df_in)}")
    print(f"Output rows: {len(df_out)}")

    numeric = df_out.select_dtypes(include=[np.number])
    if not numeric.empty:
        missing = int(numeric.isna().sum().sum())
        print(f"Missing values (numeric cols): {missing}")
        print("Summary statistics:")
        print(numeric.describe().T[["mean", "std", "min", "max"]])
    else:
        print("Missing values: n/a (no numeric columns)")

    print(f"File written to: {file_path}")
    logger.info("%s -> %s (rows=%d)", label, file_path, len(df_out))

## Cell: Dummy data generation — study area and administrative boundary

All synthetic datasets share the projected coordinate system **EPSG:32646** and are built from
the *same* geometric skeleton (a 5 km × 5 km study area split into a 3×3 ward grid), which is
what guarantees that roads intersect the flood raster, facilities sit inside the boundary,
and travel origins/destinations sit on the road network — everything is generated from one
spatially consistent source of truth rather than independently.

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_boundary
# Purpose : Create a synthetic administrative boundary made of 9 wards
#           (3x3 grid) covering the study area.
# Inputs  : output_shp (str), extent (tuple), n_x (int), n_y (int)
# Output  : Polygon shapefile with fields [ward_id, ward_name]
# Returns : (output_shp path, GeoDataFrame)
# ---------------------------------------------------------------------------
CRS = "EPSG:32646"
STUDY_EXTENT = (0, 0, 5000, 5000)  # minx, miny, maxx, maxy (metres)


def generate_boundary(
    output_shp: str,
    extent: tuple = STUDY_EXTENT,
    n_x: int = 3,
    n_y: int = 3,
    seed: int = 42,
) -> tuple:
    """Generate a synthetic ward-boundary polygon shapefile.

    Parameters
    ----------
    output_shp : str
        Destination path for the boundary shapefile.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    n_x, n_y : int
        Number of wards along each axis (n_x * n_y wards total).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple(str, geopandas.GeoDataFrame)
        Path to the written shapefile and the GeoDataFrame itself.
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    dx, dy = (maxx - minx) / n_x, (maxy - miny) / n_y

    polys, ward_ids, ward_names = [], [], []
    wid = 1
    for i in range(n_x):
        for j in range(n_y):
            polys.append(box(minx + i * dx, miny + j * dy, minx + (i + 1) * dx, miny + (j + 1) * dy))
            ward_ids.append(f"W{wid:02d}")
            ward_names.append(f"Ward {wid}")
            wid += 1

    gdf = gpd.GeoDataFrame({"ward_id": ward_ids, "ward_name": ward_names}, geometry=polys, crs=CRS)
    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp, gdf


boundary_path, boundary_gdf = generate_boundary(str(PROJECT["boundary"] / "boundary.shp"))
validate_output("Generate Boundary", boundary_gdf, boundary_path)

## Cell: Dummy data generation — road network

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_roads
# Purpose : Create a synthetic road network (grid + residential connectors)
#           that spans the full study extent, guaranteeing it intersects
#           both the boundary and the flood raster generated later.
# Inputs  : output_shp (str), boundary_gdf (GeoDataFrame), extent (tuple)
# Output  : LineString shapefile, fields [road_id, road_type, length_m,
#           speed_lim, lanes, status]
# Returns : (output_shp path, GeoDataFrame)
# ---------------------------------------------------------------------------
def generate_roads(
    output_shp: str,
    extent: tuple = STUDY_EXTENT,
    n_grid_x: int = 3,
    n_grid_y: int = 3,
    n_connectors: int = 12,
    seed: int = 42,
) -> tuple:
    """Generate a synthetic road network covering the study extent.

    Parameters
    ----------
    output_shp : str
        Destination path for the road shapefile.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    n_grid_x, n_grid_y : int
        Number of grid lines along each axis (aligned with ward boundaries).
    n_connectors : int
        Number of extra residential connector segments to add for density.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple(str, geopandas.GeoDataFrame)
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    dx, dy = (maxx - minx) / n_grid_x, (maxy - miny) / n_grid_y
    road_types = ["primary", "secondary", "residential"]

    records = []
    rid = 1
    for j in range(n_grid_y + 1):
        y = miny + j * dy
        records.append((f"R{rid:03d}", np.random.choice(road_types), LineString([(minx, y), (maxx, y)])))
        rid += 1
    for i in range(n_grid_x + 1):
        x = minx + i * dx
        records.append((f"R{rid:03d}", np.random.choice(road_types), LineString([(x, miny), (x, maxy)])))
        rid += 1
    for _ in range(n_connectors):
        x1, y1 = np.random.uniform(minx, maxx), np.random.uniform(miny, maxy)
        x2 = min(max(x1 + np.random.uniform(-500, 500), minx), maxx)
        y2 = min(max(y1 + np.random.uniform(-500, 500), miny), maxy)
        records.append((f"R{rid:03d}", "residential", LineString([(x1, y1), (x2, y2)])))
        rid += 1

    road_ids = [r[0] for r in records]
    r_types = [r[1] for r in records]
    geoms = [r[2] for r in records]
    lengths = [g.length for g in geoms]
    speed_lim = [int(np.random.choice([80, 60, 40, 30])) for _ in records]
    lanes = [int(np.random.choice([4, 2, 2, 1])) for _ in records]
    status = ["unknown"] * len(records)

    gdf = gpd.GeoDataFrame(
        {
            "road_id": road_ids,
            "road_type": r_types,
            "length_m": lengths,
            "speed_lim": speed_lim,
            "lanes": lanes,
            "status": status,
        },
        geometry=geoms,
        crs=CRS,
    )
    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp, gdf


roads_path, roads_gdf = generate_roads(str(PROJECT["roads"] / "roads.shp"))
validate_output("Generate Roads", roads_gdf, roads_path)

## Cell: Dummy data generation — flood raster

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_flood_raster
# Purpose : Create a synthetic flood-depth GeoTIFF (10 m resolution) whose
#           footprint exactly matches the study extent, guaranteeing every
#           road segment overlaps flooded pixels somewhere along its length.
# Inputs  : output_tif (str), extent (tuple), resolution (int, metres)
# Output  : Float32 GeoTIFF, single band, flood depth in metres
# Returns : output_tif path
# ---------------------------------------------------------------------------
def generate_flood_raster(
    output_tif: str,
    extent: tuple = STUDY_EXTENT,
    resolution: int = 10,
    seed: int = 42,
) -> str:
    """Generate a synthetic flood-depth raster covering the study extent.

    A diagonal high-depth corridor (simulating an overflowing river) is
    combined with random noise to produce a spatially realistic depth
    surface, ranging roughly 0-1.5 m.

    Parameters
    ----------
    output_tif : str
        Destination path for the GeoTIFF.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    resolution : int
        Pixel size in metres.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    str
        Path to the written raster.
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = from_origin(minx, maxy, resolution, resolution)

    xs = np.linspace(minx, maxx, width)
    ys = np.linspace(maxy, miny, height)
    xx, yy = np.meshgrid(xs, ys)

    dist_to_river = np.abs(xx - yy)  # proxy distance from the diagonal "river"
    depth = np.clip(1.2 - dist_to_river / 2000, 0, None) + np.random.rand(height, width) * 0.15
    depth = depth.astype("float32")

    Path(output_tif).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        output_tif, "w", driver="GTiff", height=height, width=width,
        count=1, dtype="float32", crs=CRS, transform=transform, nodata=-9999,
    ) as dst:
        dst.write(depth, 1)

    return output_tif


flood_path = generate_flood_raster(str(PROJECT["flood"] / "flood_depth.tif"))

with rasterio.open(flood_path) as src:
    arr = src.read(1)
print("\n--- Validation: Generate Flood Raster ---")
print(f"Raster shape : {arr.shape}")
print(f"Depth range  : {arr.min():.2f} m to {arr.max():.2f} m")
print(f"Mean depth   : {arr.mean():.2f} m")
print(f"File written to: {flood_path}")
logger.info("Generate Flood Raster -> %s (shape=%s)", flood_path, arr.shape)

## Cell: Dummy data generation — essential facilities

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_facilities
# Purpose : Create synthetic essential-facility points, constrained to lie
#           strictly inside the study area (and therefore inside the union
#           of ward polygons).
# Inputs  : output_shp (str), boundary_gdf (GeoDataFrame), n_facilities (int)
# Output  : Point shapefile, fields [fac_id, fac_type]
# Returns : (output_shp path, GeoDataFrame)
# ---------------------------------------------------------------------------
def generate_facilities(
    output_shp: str,
    extent: tuple = STUDY_EXTENT,
    n_facilities: int = 25,
    seed: int = 42,
) -> tuple:
    """Generate synthetic essential-facility points inside the study area.

    Parameters
    ----------
    output_shp : str
        Destination path for the facilities shapefile.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    n_facilities : int
        Number of facility points to generate.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple(str, geopandas.GeoDataFrame)
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    fac_types = ["Hospital", "School", "Market", "Bus Stop", "Metro Station"]

    fx = np.random.uniform(minx + 50, maxx - 50, n_facilities)
    fy = np.random.uniform(miny + 50, maxy - 50, n_facilities)

    gdf = gpd.GeoDataFrame(
        {
            "fac_id": [f"F{i+1:03d}" for i in range(n_facilities)],
            "fac_type": [np.random.choice(fac_types) for _ in range(n_facilities)],
        },
        geometry=[Point(x, y) for x, y in zip(fx, fy)],
        crs=CRS,
    )
    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp, gdf


facilities_path, facilities_gdf = generate_facilities(str(PROJECT["facilities"] / "facilities.shp"))
validate_output("Generate Facilities", facilities_gdf, facilities_path)

## Cell: Dummy data generation — travel survey

Trip origins and destinations are sampled from the **actual vertices of the road network**
generated above (with small positional jitter), so every trip starts and ends on or near a real
road segment, and each trip's `ward_id` is assigned by a true point-in-polygon test against the
boundary layer.

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_travel_survey
# Purpose : Create a synthetic household travel survey whose origins and
#           destinations are drawn from real road-network vertices, and
#           whose ward_id is assigned via point-in-polygon against the
#           boundary layer, guaranteeing spatial consistency with the
#           other datasets.
# Inputs  : output_csv (str), roads_gdf (GeoDataFrame), boundary_gdf (GeoDataFrame)
# Output  : CSV, fields per the project data dictionary
# Returns : output_csv path
# ---------------------------------------------------------------------------
def generate_travel_survey(
    output_csv: str,
    roads_gdf: gpd.GeoDataFrame,
    boundary_gdf: gpd.GeoDataFrame,
    n_trips: int = 200,
    seed: int = 42,
) -> str:
    """Generate a synthetic travel survey consistent with the road network.

    Parameters
    ----------
    output_csv : str
        Destination path for the travel-survey CSV.
    roads_gdf : geopandas.GeoDataFrame
        The road network used to source realistic origin/destination points.
    boundary_gdf : geopandas.GeoDataFrame
        Ward polygons, used to assign each trip's ward_id.
    n_trips : int
        Number of synthetic trips to generate.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    str
        Path to the written CSV.
    """
    np.random.seed(seed)
    all_coords = np.array([pt for geom in roads_gdf.geometry for pt in geom.coords])

    modes = ["car", "bus", "walk", "rickshaw", "motorbike"]
    idx_o = np.random.choice(len(all_coords), n_trips)
    idx_d = np.random.choice(len(all_coords), n_trips)

    rows = []
    for t in range(n_trips):
        ox, oy = all_coords[idx_o[t]] + np.random.normal(0, 20, 2)
        dx_, dy_ = all_coords[idx_d[t]] + np.random.normal(0, 20, 2)
        planned = np.random.choice(modes)
        actual = planned if np.random.rand() > 0.25 else np.random.choice(modes)
        tt_before = np.random.uniform(5, 60)
        tt_after = tt_before * np.random.uniform(1.0, 2.5)
        cancelled = bool(np.random.rand() < 0.1)
        route_changed = bool(np.random.rand() < 0.3)

        pt = Point(ox, oy)
        match = boundary_gdf[boundary_gdf.contains(pt)]
        ward_id = match.iloc[0]["ward_id"] if len(match) > 0 else np.random.choice(boundary_gdf["ward_id"])

        rows.append([
            f"T{t+1:04d}", f"P{np.random.randint(1, 150):04d}", ox, oy, dx_, dy_,
            planned, actual, round(tt_before, 1), round(tt_after, 1),
            cancelled, route_changed, ward_id,
        ])

    df = pd.DataFrame(rows, columns=[
        "trip_id", "person_id", "origin_x", "origin_y", "destination_x", "destination_y",
        "planned_mode", "actual_mode", "travel_time_before", "travel_time_after",
        "trip_cancelled", "route_changed", "ward_id",
    ])
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return output_csv


travel_path = generate_travel_survey(
    str(PROJECT["travel"] / "travel_survey.csv"), roads_gdf, boundary_gdf
)
travel_df = pd.read_csv(travel_path)
validate_output("Generate Travel Survey", travel_df, travel_path)

## Cell: Part 1 summary

All five synthetic datasets have now been generated and written to disk under
`/content/project/input/`, and are spatially consistent with one another (shared CRS,
shared study extent, roads sourced from the same skeleton as trip origins/destinations,
facilities and boundary sharing the same footprint).

**Files on disk:**
- `input/boundary/boundary.shp`
- `input/roads/roads.shp`
- `input/flood/flood_depth.tif`
- `input/facilities/facilities.shp`
- `input/travel/travel_survey.csv`

**Reusable objects still in memory for convenience** (though every downstream function in Parts
2-4 will read from disk, not from these variables, per the project's file-based-pipeline
requirement): `PROJECT`, `boundary_gdf`, `roads_gdf`, `facilities_gdf`, `travel_df`, `logger`,
`validate_output()`.

**Next: Part 2** — Road-Flood Overlay → Road Status Classification → Infrastructure
Functionality (IF) → Accessibility Index (AI).


# Part 2 — Infrastructure Functionality (IF) and Accessibility (AI)

This part implements the second and third stages of the workflow:

```
Road Network ─┐
              ├──► Road-Flood Overlay ──► Road Status ──► Infrastructure Functionality (IF)
Flood Raster ─┘                                                     │
                                                                     ▼
Essential Facilities ──────────────────────────────────► Accessibility Index (AI)
```

Every function below reads its inputs from disk (the files written in Part 1) and writes its
output back to disk, so this part can be re-run independently as long as Part 1's `input/`
folder exists.


## Function 1 — `road_flood_overlay()`

**Purpose:** Compute the average flood depth experienced by each road segment.

**Inputs:** `road_shp` (Shapefile, LineString) · `flood_raster` (GeoTIFF) · `output_shp` (str)

**Input file types:** `.shp` (roads), `.tif` (flood depth)

**Output:** Shapefile identical to the input road network plus one new field, `avg_flood`
(average flood depth in metres intersecting each segment).

**Output file type:** `.shp`

**Returned value:** Path to the output shapefile (str)

**Processing steps:**
1. Read the road shapefile and the flood raster's pixel resolution.
2. Because a perfectly horizontal or vertical `LineString` has a zero-area bounding box (which
   breaks raster rasterization), buffer each line by half a pixel width purely for the purpose
   of sampling — the *original*, unbuffered line geometry is what gets saved to the output.
3. Use `rasterstats.zonal_stats()` (`all_touched=True`) to compute the mean flood depth under
   each buffered segment.
4. Attach the result as `avg_flood` and write the shapefile.

**Dependencies:** `geopandas`, `rasterstats`, `rasterio`, `pathlib`

**Example usage:** `road_flood_overlay("input/roads/roads.shp", "input/flood/flood_depth.tif", "intermediate/01_overlay/road_overlay.shp")`


In [ ]:
def road_flood_overlay(road_shp: str, flood_raster: str, output_shp: str) -> str:
    """Compute average flood depth per road segment via raster zonal statistics.

    Parameters
    ----------
    road_shp : str
        Path to the road-network shapefile (LineString).
    flood_raster : str
        Path to the flood-depth GeoTIFF.
    output_shp : str
        Destination path for the overlay shapefile.

    Returns
    -------
    str
        Path to the written overlay shapefile.
    """
    roads = gpd.read_file(road_shp)
    if roads.empty:
        raise ValueError(f"No road features found in {road_shp}")

    with rasterio.open(flood_raster) as src:
        res = src.res[0]

    # Buffer purely for sampling purposes; the saved geometry stays as the original line.
    buffered = roads.geometry.buffer(res / 2)
    stats = rasterstats.zonal_stats(
        buffered, flood_raster, stats=["mean"], all_touched=True, nodata=-9999
    )
    roads["avg_flood"] = [s["mean"] if s["mean"] is not None else 0.0 for s in stats]

    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    roads.to_file(output_shp)
    return output_shp


overlay_path = road_flood_overlay(
    str(PROJECT["roads"] / "roads.shp"),
    str(PROJECT["flood"] / "flood_depth.tif"),
    str(PROJECT["overlay"] / "road_overlay.shp"),
)
overlay_gdf = gpd.read_file(overlay_path)
validate_output("Road-Flood Overlay", overlay_gdf, overlay_path, df_in=roads_gdf)

## Function 2 — `classify_road_status()`

**Purpose:** Convert continuous flood depth into a discrete serviceability status per segment.

**Inputs:** `overlay_shp` (Shapefile with `avg_flood` field) · `output_shp` (str)

**Input file types:** `.shp`

**Output:** Shapefile with a new/updated `status` field: `Operational` (< 0.20 m),
`Partial` (0.20–0.50 m), `Closed` (> 0.50 m).

**Output file type:** `.shp`

**Returned value:** Path to the output shapefile (str)

**Processing steps:**
1. Read the overlay shapefile.
2. Apply the three flood-depth thresholds to `avg_flood`.
3. Write the classified result.

**Dependencies:** `geopandas`, `pathlib`

**Example usage:** `classify_road_status("intermediate/01_overlay/road_overlay.shp", "intermediate/02_status/road_status.shp")`


In [ ]:
def classify_road_status(overlay_shp: str, output_shp: str) -> str:
    """Classify each road segment as Operational / Partial / Closed by flood depth.

    Parameters
    ----------
    overlay_shp : str
        Path to the shapefile produced by ``road_flood_overlay`` (must contain
        an ``avg_flood`` field).
    output_shp : str
        Destination path for the classified shapefile.

    Returns
    -------
    str
        Path to the written shapefile.
    """
    gdf = gpd.read_file(overlay_shp)
    if "avg_flood" not in gdf.columns:
        raise KeyError("Input shapefile is missing the 'avg_flood' field.")

    def _classify(depth: float) -> str:
        if depth < 0.20:
            return "Operational"
        elif depth <= 0.50:
            return "Partial"
        return "Closed"

    gdf["status"] = gdf["avg_flood"].apply(_classify)

    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp


status_path = classify_road_status(overlay_path, str(PROJECT["status"] / "road_status.shp"))
status_gdf = gpd.read_file(status_path)
validate_output("Road Status Classification", status_gdf, status_path, df_in=overlay_gdf)
print("\nStatus breakdown:")
print(status_gdf["status"].value_counts())

## Function 3 — `calculate_if()`

**Purpose:** Aggregate road serviceability into a ward-level **Infrastructure Functionality**
score.

**Inputs:** `road_status_shp` (Shapefile) · `boundary_shp` (Shapefile) · `output_csv` (str)

**Input file types:** `.shp`, `.shp`

**Output:** CSV with fields `ward_id`, `effective_length`, `total_length`, `IF`.

**Output file type:** `.csv`

**Returned value:** Path to the output CSV (str)

**Rules:** `Operational = 100%` weight, `Partial = 50%` weight, `Closed = 0%` weight.
`IF = effective_length / total_length * 100`.

**Processing steps:**
1. Spatially split every road segment at ward boundaries (`geopandas.overlay`, intersection),
   so a road that crosses two wards contributes the correct length to each.
2. Compute each split segment's length and its weighted "effective length".
3. Aggregate by `ward_id`.
4. Wards with no road segments at all are still included, with `IF = 0`.

**Dependencies:** `geopandas`, `pandas`, `pathlib`

**Example usage:** `calculate_if("intermediate/02_status/road_status.shp", "input/boundary/boundary.shp", "intermediate/03_if/if.csv")`


In [ ]:
def calculate_if(road_status_shp: str, boundary_shp: str, output_csv: str) -> str:
    """Calculate ward-level Infrastructure Functionality (IF).

    Parameters
    ----------
    road_status_shp : str
        Path to the classified road-status shapefile (must contain ``status``).
    boundary_shp : str
        Path to the ward-boundary shapefile.
    output_csv : str
        Destination path for the IF CSV.

    Returns
    -------
    str
        Path to the written CSV.
    """
    roads = gpd.read_file(road_status_shp)
    boundary = gpd.read_file(boundary_shp)
    weights = {"Operational": 1.0, "Partial": 0.5, "Closed": 0.0}

    split = gpd.overlay(
        roads[["road_id", "status", "geometry"]],
        boundary[["ward_id", "geometry"]],
        how="intersection",
    )
    split["seg_len"] = split.geometry.length
    split["eff_len"] = split["seg_len"] * split["status"].map(weights)

    agg = (
        split.groupby("ward_id")
        .agg(effective_length=("eff_len", "sum"), total_length=("seg_len", "sum"))
        .reset_index()
    )
    agg["IF"] = (agg["effective_length"] / agg["total_length"] * 100).round(2)

    all_wards = boundary[["ward_id"]].copy()
    agg = all_wards.merge(agg, on="ward_id", how="left").fillna(0)

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    agg.to_csv(output_csv, index=False)
    return output_csv


if_path = calculate_if(
    status_path, str(PROJECT["boundary"] / "boundary.shp"), str(PROJECT["if"] / "if.csv")
)
if_df = pd.read_csv(if_path)
validate_output("Infrastructure Functionality (IF)", if_df, if_path)

## Function 4 — `calculate_accessibility()`

**Purpose:** Measure, per ward, what share of the region's essential facilities remain reachable
through the surviving (non-Closed) road network.

**Inputs:** `road_status_shp` (Shapefile) · `facilities_shp` (Shapefile) · `boundary_shp`
(Shapefile) · `output_csv` (str)

**Input file types:** `.shp` × 3

**Output:** CSV with fields `ward_id`, `reachable_facilities`, `total_facilities`, `AI`.

**Output file type:** `.csv`

**Returned value:** Path to the output CSV (str)

**Processing steps:**
1. Build a `networkx.Graph` from road segments **excluding** any segment classified `Closed`
   (closed roads cannot be traversed, per the project specification). Road-segment endpoints
   (rounded to 0.1 m) become graph nodes; edges are weighted by Euclidean segment length.
2. For each ward, take its centroid and snap it to the nearest graph node (its "access point").
3. For each facility, snap it to the nearest graph node.
4. A facility counts as **reachable** from a ward if `networkx.has_path()` finds a path between
   the ward's access node and the facility's node in the Closed-road-free graph.
5. `AI = reachable_facilities / total_facilities * 100`.

**Dependencies:** `geopandas`, `networkx`, `numpy`, `pandas`, `pathlib`

**Example usage:** `calculate_accessibility("intermediate/02_status/road_status.shp", "input/facilities/facilities.shp", "input/boundary/boundary.shp", "intermediate/04_accessibility/accessibility.csv")`


In [ ]:
def build_road_graph(roads_gdf: gpd.GeoDataFrame, exclude_closed: bool = True) -> nx.Graph:
    """Build an undirected graph from road-segment geometries.

    Parameters
    ----------
    roads_gdf : geopandas.GeoDataFrame
        Road segments with a ``status`` field.
    exclude_closed : bool
        If True, segments with status == "Closed" are skipped entirely,
        i.e. they cannot be traversed.

    Returns
    -------
    networkx.Graph
        Nodes are rounded (x, y) coordinates; edges are weighted by length.
    """
    G = nx.Graph()
    for _, row in roads_gdf.iterrows():
        if exclude_closed and row["status"] == "Closed":
            continue
        coords = list(row.geometry.coords)
        for i in range(len(coords) - 1):
            u = (round(coords[i][0], 1), round(coords[i][1], 1))
            v = (round(coords[i + 1][0], 1), round(coords[i + 1][1], 1))
            w = ((u[0] - v[0]) ** 2 + (u[1] - v[1]) ** 2) ** 0.5
            G.add_edge(u, v, weight=w)
    return G


def _nearest_node(G: nx.Graph, point) -> Optional[tuple]:
    """Return the graph node closest to a shapely Point (Euclidean distance)."""
    nodes = list(G.nodes)
    if not nodes:
        return None
    dists = [(n[0] - point.x) ** 2 + (n[1] - point.y) ** 2 for n in nodes]
    return nodes[int(np.argmin(dists))]


def calculate_accessibility(
    road_status_shp: str, facilities_shp: str, boundary_shp: str, output_csv: str
) -> str:
    """Calculate ward-level Accessibility Index (AI) using the surviving road network.

    Parameters
    ----------
    road_status_shp : str
        Path to the classified road-status shapefile.
    facilities_shp : str
        Path to the essential-facilities point shapefile.
    boundary_shp : str
        Path to the ward-boundary shapefile.
    output_csv : str
        Destination path for the accessibility CSV.

    Returns
    -------
    str
        Path to the written CSV.
    """
    roads = gpd.read_file(road_status_shp)
    facilities = gpd.read_file(facilities_shp)
    boundary = gpd.read_file(boundary_shp)

    G_pass = build_road_graph(roads, exclude_closed=True)
    total_facilities = len(facilities)

    rows = []
    for _, ward in boundary.iterrows():
        centroid = ward.geometry.centroid
        ward_node = _nearest_node(G_pass, centroid)
        reachable = 0
        if ward_node is not None:
            for _, fac in facilities.iterrows():
                fac_node = _nearest_node(G_pass, fac.geometry)
                if fac_node is not None and nx.has_path(G_pass, ward_node, fac_node):
                    reachable += 1
        AI = round(reachable / total_facilities * 100, 2) if total_facilities > 0 else 0.0
        rows.append([ward["ward_id"], reachable, total_facilities, AI])

    df = pd.DataFrame(rows, columns=["ward_id", "reachable_facilities", "total_facilities", "AI"])
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return output_csv


accessibility_path = calculate_accessibility(
    status_path,
    str(PROJECT["facilities"] / "facilities.shp"),
    str(PROJECT["boundary"] / "boundary.shp"),
    str(PROJECT["accessibility"] / "accessibility.csv"),
)
accessibility_df = pd.read_csv(accessibility_path)
validate_output("Accessibility Index (AI)", accessibility_df, accessibility_path)

## Part 2 summary

Two ward-level indices have now been computed and written to disk:

- `intermediate/03_if/if.csv` — Infrastructure Functionality (IF)
- `intermediate/04_accessibility/accessibility.csv` — Accessibility Index (AI)

Both are keyed by `ward_id`, ready to be joined with Behavioural Adaptation (BA) in Part 3 to
build the master resilience dataset and the final TRI.

**Next: Part 3** — Travel Behaviour → Behavioural Adaptation (BA) → Master Dataset →
Transport Resilience Index (TRI).


# Part 3 — Behavioural Adaptation, Master Dataset, and the Transport Resilience Index

This part implements the remaining stages of the workflow:

```
Travel Survey ──► Travel Behaviour Metrics ──► Behavioural Adaptation (BA)
                                                          │
                            IF  +  AI  +  BA  ───────────►│
                                                          ▼
                                          Master Dataset ──► TRI
```

By the end of this part, every ward in the study area will have a single historical
**Transport Resilience Index** value, built entirely from the IF (Part 2), AI (Part 2), and
BA (this part) scores — no forecasting, no learned models, just transparent aggregation.


## Function 5 — `calculate_travel_behaviour()`

**Purpose:** Convert raw travel-survey responses into per-trip behavioural indicators.

**Inputs:** `travel_csv` (CSV, the raw travel survey) · `output_csv` (str)

**Input file types:** `.csv`

**Output:** CSV with `trip_id`, `person_id`, `ward_id`, `delay_minutes`, `mode_changed`,
`trip_completed`, `route_changed`.

**Output file type:** `.csv`

**Returned value:** Path to the output CSV (str)

**Processing steps:**
1. Read the raw travel survey.
2. `delay_minutes = travel_time_after - travel_time_before`.
3. `mode_changed = planned_mode != actual_mode`.
4. `trip_completed = NOT trip_cancelled`.
5. Carry `route_changed` through as a clean boolean.

**Dependencies:** `pandas`, `pathlib`

**Example usage:** `calculate_travel_behaviour("input/travel/travel_survey.csv", "intermediate/05_behaviour/behaviour.csv")`


In [ ]:
def calculate_travel_behaviour(travel_csv: str, output_csv: str) -> str:
    """Derive per-trip behavioural indicators from the raw travel survey.

    Parameters
    ----------
    travel_csv : str
        Path to the raw travel-survey CSV.
    output_csv : str
        Destination path for the derived behaviour CSV.

    Returns
    -------
    str
        Path to the written CSV.
    """
    df = pd.read_csv(travel_csv)
    required = {"travel_time_before", "travel_time_after", "planned_mode", "actual_mode",
                "trip_cancelled", "route_changed", "ward_id"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Travel survey is missing required fields: {missing}")

    df["delay_minutes"] = (df["travel_time_after"] - df["travel_time_before"]).round(2)
    df["mode_changed"] = df["planned_mode"] != df["actual_mode"]
    df["trip_completed"] = ~df["trip_cancelled"].astype(bool)
    df["route_changed"] = df["route_changed"].astype(bool)

    out = df[["trip_id", "person_id", "ward_id", "delay_minutes", "mode_changed",
              "trip_completed", "route_changed"]]
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_csv, index=False)
    return output_csv


behaviour_path = calculate_travel_behaviour(
    str(PROJECT["travel"] / "travel_survey.csv"), str(PROJECT["behaviour"] / "behaviour.csv")
)
behaviour_df = pd.read_csv(behaviour_path)
validate_output("Travel Behaviour", behaviour_df, behaviour_path, df_in=travel_df)

## Function 6 — `calculate_behavioural_adaptation()`

**Purpose:** Aggregate per-trip behaviour into a single, transparent, ward-level
**Behavioural Adaptation** score describing how well travellers coped with the disruption.

**Inputs:** `behaviour_csv` (CSV) · `output_csv` (str)

**Input file types:** `.csv`

**Output:** CSV with `ward_id`, `avg_delay`, `pct_completed`, `pct_mode_changed`,
`pct_route_changed`, `BA` (normalized 0–1).

**Output file type:** `.csv`

**Returned value:** Path to the output CSV (str)

**Rule-based scoring (fully transparent, no learned weights):** for each ward we compute three
component scores, each on a 0–1 scale where **1 = best (least disrupted)**, then average them:

1. `completion_score` — share of trips completed (not cancelled).
2. `delay_score` — `1 - (ward's average delay / the worst average delay observed across all
   wards)`, so the most-delayed ward scores 0 and an undelayed ward scores 1.
3. `stability_score` — `1 - average(% trips that changed mode, % trips that changed route) / 100`,
   so wards where more people were forced to change mode/route score lower.

`BA = mean(completion_score, delay_score, stability_score)`, clipped to `[0, 1]`.

**Processing steps:**
1. Group `behaviour.csv` by `ward_id`.
2. Compute the four raw aggregate statistics.
3. Compute the three component scores and their mean, `BA`.

**Dependencies:** `pandas`, `pathlib`

**Example usage:** `calculate_behavioural_adaptation("intermediate/05_behaviour/behaviour.csv", "intermediate/06_ba/ba.csv")`


In [ ]:
def calculate_behavioural_adaptation(behaviour_csv: str, output_csv: str) -> str:
    """Aggregate trip-level behaviour into a ward-level Behavioural Adaptation score.

    Parameters
    ----------
    behaviour_csv : str
        Path to the CSV produced by ``calculate_travel_behaviour``.
    output_csv : str
        Destination path for the BA CSV.

    Returns
    -------
    str
        Path to the written CSV.
    """
    df = pd.read_csv(behaviour_csv)

    agg = df.groupby("ward_id").agg(
        avg_delay=("delay_minutes", "mean"),
        pct_completed=("trip_completed", lambda s: s.mean() * 100),
        pct_mode_changed=("mode_changed", lambda s: s.mean() * 100),
        pct_route_changed=("route_changed", lambda s: s.mean() * 100),
        n_trips=("trip_id", "count"),
    ).reset_index()

    max_delay = agg["avg_delay"].max() if agg["avg_delay"].max() > 0 else 1.0
    agg["completion_score"] = agg["pct_completed"] / 100.0
    agg["delay_score"] = 1 - (agg["avg_delay"].clip(lower=0) / max_delay).clip(0, 1)
    agg["stability_score"] = 1 - ((agg["pct_mode_changed"] + agg["pct_route_changed"]) / 200.0)

    agg["BA"] = (
        (agg["completion_score"] + agg["delay_score"] + agg["stability_score"]) / 3
    ).clip(0, 1).round(3)

    out = agg[["ward_id", "avg_delay", "pct_completed", "pct_mode_changed", "pct_route_changed", "BA"]]
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_csv, index=False)
    return output_csv


ba_path = calculate_behavioural_adaptation(behaviour_path, str(PROJECT["ba"] / "ba.csv"))
ba_df = pd.read_csv(ba_path)
validate_output("Behavioural Adaptation (BA)", ba_df, ba_path)

## Function 7 — `build_master_dataset()`

**Purpose:** Join IF, AI, and BA into a single ward-level table ready for the final TRI
calculation.

**Inputs:** `if_csv` · `accessibility_csv` · `ba_csv` (all CSV) · `output_csv` (str)

**Input file types:** `.csv` × 3

**Output:** CSV with `ward_id`, `IF`, `AI`, `BA`.

**Output file type:** `.csv`

**Returned value:** Path to the output CSV (str)

**Note on units:** `IF` and `AI` are already expressed on a 0–100 scale. `BA` was deliberately
normalized to 0–1 in Function 6 (per the project specification), so here it is rescaled ×100
to put all three indices on the same 0–100 basis before they are averaged into TRI — otherwise
`(IF + AI + BA) / 3` would be dominated by IF and AI.

**Processing steps:**
1. Read the three CSVs, keeping only the relevant columns.
2. Outer-merge on `ward_id` so no ward is silently dropped.
3. Rescale `BA` to 0–100.
4. Fill any missing values with 0 (a ward absent from one metric had no measurable signal there).

**Dependencies:** `pandas`, `pathlib`

**Example usage:** `build_master_dataset("intermediate/03_if/if.csv", "intermediate/04_accessibility/accessibility.csv", "intermediate/06_ba/ba.csv", "intermediate/07_master/master_resilience.csv")`


In [ ]:
def build_master_dataset(if_csv: str, accessibility_csv: str, ba_csv: str, output_csv: str) -> str:
    """Merge IF, AI, and BA into a single master resilience table.

    Parameters
    ----------
    if_csv : str
        Path to the Infrastructure Functionality CSV.
    accessibility_csv : str
        Path to the Accessibility Index CSV.
    ba_csv : str
        Path to the Behavioural Adaptation CSV.
    output_csv : str
        Destination path for the master dataset.

    Returns
    -------
    str
        Path to the written CSV.
    """
    if_df = pd.read_csv(if_csv)[["ward_id", "IF"]]
    ai_df = pd.read_csv(accessibility_csv)[["ward_id", "AI"]]
    ba_df = pd.read_csv(ba_csv)[["ward_id", "BA"]].copy()
    ba_df["BA"] = (ba_df["BA"] * 100).round(2)  # rescale 0-1 -> 0-100 to match IF/AI

    master = if_df.merge(ai_df, on="ward_id", how="outer").merge(ba_df, on="ward_id", how="outer")
    master = master.fillna(0)

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    master.to_csv(output_csv, index=False)
    return output_csv


master_path = build_master_dataset(
    if_path, accessibility_path, ba_path, str(PROJECT["master"] / "master_resilience.csv")
)
master_df = pd.read_csv(master_path)
validate_output("Master Dataset", master_df, master_path)

## Function 8 — `calculate_tri()`

**Purpose:** Compute the final ward-level **Transport Resilience Index**.

**Inputs:** `master_csv` (CSV) · `output_csv` (str)

**Input file types:** `.csv`

**Output:** CSV with `ward_id`, `IF`, `AI`, `BA`, `TRI`.

**Output file type:** `.csv`

**Returned value:** Path to the output CSV (str)

**Formula:** `TRI = (IF + AI + BA) / 3` (simple, transparent, equal-weighted average — no
fitted or learned weighting).

**Processing steps:**
1. Read the master dataset.
2. Apply the formula row-wise.
3. Write the result.

**Dependencies:** `pandas`, `pathlib`

**Example usage:** `calculate_tri("intermediate/07_master/master_resilience.csv", "intermediate/08_tri/tri.csv")`


In [ ]:
def calculate_tri(master_csv: str, output_csv: str) -> str:
    """Calculate the final Transport Resilience Index (TRI) per ward.

    Parameters
    ----------
    master_csv : str
        Path to the master resilience dataset (must contain IF, AI, BA).
    output_csv : str
        Destination path for the TRI CSV.

    Returns
    -------
    str
        Path to the written CSV.
    """
    df = pd.read_csv(master_csv)
    for col in ("IF", "AI", "BA"):
        if col not in df.columns:
            raise KeyError(f"Master dataset is missing required column '{col}'.")

    df["TRI"] = ((df["IF"] + df["AI"] + df["BA"]) / 3).round(2)

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return output_csv


tri_path = calculate_tri(master_path, str(PROJECT["tri"] / "tri.csv"))
tri_df = pd.read_csv(tri_path)
validate_output("Transport Resilience Index (TRI)", tri_df, tri_path)
print("\nFinal ward-level TRI:")
print(tri_df.sort_values("TRI", ascending=False).to_string(index=False))

## Part 3 summary

The complete historical resilience chain is now on disk:

- `intermediate/05_behaviour/behaviour.csv`
- `intermediate/06_ba/ba.csv`
- `intermediate/07_master/master_resilience.csv`
- `intermediate/08_tri/tri.csv` — **the final TRI per ward**

Each ward now has a single, auditable resilience score built purely from historical
observations of what happened to the road network and to travellers during the event —
no predictive modelling anywhere in the chain.

**Next: Part 4** — Visualization (`visualize_vector`, `visualize_raster`, `visualize_csv`,
`visualize_project_outputs`), the full validation report, and the end-to-end `run_pipeline()`
function that ties every part together into one automatic execution.


# Part 4 — Visualization, Validation, and the End-to-End Pipeline

This final part adds four reusable visualization functions, then wires everything from Parts
1–3 into a single `run_pipeline()` entry point that regenerates the dummy data and recomputes
IF → AI → BA → TRI automatically, from scratch, with no manual steps.


## `visualize_vector()`

**Purpose:** Produce a standard, presentation-ready map for any shapefile (roads, boundary,
facilities), automatically colour-coding by a categorical field when one is present
(`status`, `fac_type`, or `ward_name`).

**Inputs:** `shp_path` (str) · `output_png` (str) · `title` (str, optional) ·
`category_field` (str, optional — auto-detected if omitted)

**Output:** PNG map with title, legend, grid, and a north arrow (the plot is already
north-up because projected x/y coordinates are used directly).

**Returns:** Path to the saved PNG (str)


In [ ]:
def visualize_vector(
    shp_path: str, output_png: str, title: Optional[str] = None, category_field: Optional[str] = None
) -> str:
    """Render any shapefile as a labelled, north-up map and save it as a PNG.

    Parameters
    ----------
    shp_path : str
        Path to the shapefile to visualize.
    output_png : str
        Destination path for the rendered PNG.
    title : str, optional
        Map title. Defaults to the shapefile's stem.
    category_field : str, optional
        Field to colour-code by. If omitted, auto-detects among
        {"status", "fac_type", "ward_name"}.

    Returns
    -------
    str
        Path to the written PNG.
    """
    gdf = gpd.read_file(shp_path)
    fig, ax = plt.subplots(figsize=(7, 7))

    if category_field is None:
        for cand in ["status", "fac_type", "ward_name"]:
            if cand in gdf.columns:
                category_field = cand
                break

    if category_field and category_field in gdf.columns:
        import matplotlib.patches as mpatches
        categories = gdf[category_field].unique()
        cmap = plt.get_cmap("tab10")
        handles = []
        for i, cat in enumerate(categories):
            sub = gdf[gdf[category_field] == cat]
            color = cmap(i % 10)
            sub.plot(ax=ax, color=color, linewidth=2, markersize=30)
            handles.append(mpatches.Patch(color=color, label=str(cat)))
        ax.legend(handles=handles, title=category_field, loc="upper right", fontsize=8)
    else:
        gdf.plot(ax=ax, color="steelblue", linewidth=2, markersize=30)

    ax.set_title(title or Path(shp_path).stem, fontsize=13, fontweight="bold")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.annotate(
        "N", xy=(0.95, 0.95), xytext=(0.95, 0.85), xycoords="axes fraction",
        arrowprops=dict(facecolor="black", width=3, headwidth=8), ha="center", fontsize=10,
    )
    ax.set_aspect("equal")

    Path(output_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=120)
    plt.close(fig)
    return output_png


sample_map = visualize_vector(status_path, str(PROJECT["maps"] / "road_status.png"), "Road Status After Flood")
print(f"Saved: {sample_map}")

## `visualize_raster()`

**Purpose:** Produce a standard, presentation-ready map for any single-band raster (here, the
flood-depth GeoTIFF), with a colourbar, title, and grid.

**Inputs:** `tif_path` (str) · `output_png` (str) · `title` (str, optional)

**Output:** PNG raster map.

**Returns:** Path to the saved PNG (str)


In [ ]:
def visualize_raster(tif_path: str, output_png: str, title: Optional[str] = None) -> str:
    """Render a single-band raster as a colour map with a colourbar and save it as a PNG.

    Parameters
    ----------
    tif_path : str
        Path to the GeoTIFF to visualize.
    output_png : str
        Destination path for the rendered PNG.
    title : str, optional
        Map title. Defaults to the raster's stem.

    Returns
    -------
    str
        Path to the written PNG.
    """
    with rasterio.open(tif_path) as src:
        arr = src.read(1)
        arr = np.ma.masked_equal(arr, src.nodata) if src.nodata is not None else arr
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(arr, cmap="Blues", extent=extent, origin="upper")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Flood depth (m)")
    ax.set_title(title or Path(tif_path).stem, fontsize=13, fontweight="bold")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.grid(True, linestyle="--", alpha=0.4)

    Path(output_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=120)
    plt.close(fig)
    return output_png


sample_raster_map = visualize_raster(
    str(PROJECT["flood"] / "flood_depth.tif"), str(PROJECT["maps"] / "flood_depth.png"), "Historical Flood Depth"
)
print(f"Saved: {sample_raster_map}")

## `visualize_csv()`

**Purpose:** Produce a single-image summary of any tabular output: first 10 rows, a histogram
of its leading numeric column, and either a missing-values bar chart or a category-count bar
chart (whichever is informative for that file). Summary statistics are also printed to the
console.

**Inputs:** `csv_path` (str) · `output_png` (str) · `title` (str, optional)

**Output:** PNG summary panel; printed summary statistics and missing-value counts.

**Returns:** Path to the saved PNG (str)


In [ ]:
def visualize_csv(csv_path: str, output_png: str, title: Optional[str] = None) -> str:
    """Render a tabular CSV summary (head, histogram, missing/category bar chart) as a PNG.

    Parameters
    ----------
    csv_path : str
        Path to the CSV to visualize.
    output_png : str
        Destination path for the rendered PNG.
    title : str, optional
        Panel title. Defaults to the CSV's stem.

    Returns
    -------
    str
        Path to the written PNG.
    """
    df = pd.read_csv(csv_path)
    numeric = df.select_dtypes(include=[np.number])

    fig = plt.figure(figsize=(11, 8))
    gs = fig.add_gridspec(2, 2)

    ax0 = fig.add_subplot(gs[0, :])
    ax0.axis("off")
    head = df.head(10)
    tbl = ax0.table(cellText=head.values, colLabels=head.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 1.3)
    ax0.set_title(f"{title or Path(csv_path).stem} — first 10 rows", fontsize=11, fontweight="bold")

    ax1 = fig.add_subplot(gs[1, 0])
    if not numeric.empty:
        numeric.iloc[:, 0].plot(kind="hist", ax=ax1, bins=15, color="teal", edgecolor="black")
        ax1.set_title(f"Histogram: {numeric.columns[0]}", fontsize=10)
    else:
        ax1.text(0.5, 0.5, "No numeric columns", ha="center")
        ax1.axis("off")

    ax2 = fig.add_subplot(gs[1, 1])
    missing = df.isna().sum()
    if missing.sum() > 0:
        missing[missing > 0].plot(kind="bar", ax=ax2, color="indianred")
        ax2.set_title("Missing values by column", fontsize=10)
    else:
        cat_cols = df.select_dtypes(exclude=[np.number]).columns
        if len(cat_cols) > 0:
            df[cat_cols[0]].value_counts().plot(kind="bar", ax=ax2, color="darkorange")
            ax2.set_title(f"Counts: {cat_cols[0]}", fontsize=10)
        else:
            ax2.text(0.5, 0.5, "No missing values", ha="center")
            ax2.axis("off")

    Path(output_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=120)
    plt.close(fig)

    print(f"\n--- {title or Path(csv_path).stem}: summary statistics ---")
    if not numeric.empty:
        print(numeric.describe().T[["mean", "std", "min", "max"]])
    print(f"Missing values total: {int(df.isna().sum().sum())}")
    return output_png


sample_csv_map = visualize_csv(tri_path, str(PROJECT["maps"] / "tri_summary.png"), "Transport Resilience Index")
print(f"Saved: {sample_csv_map}")

## `visualize_project_outputs()`

**Purpose:** Automatically scan the project folders, generate an individual PNG for every
shapefile, raster, and CSV produced by the pipeline (reusing the three functions above), and
assemble them into a single contact-sheet grid (auto-sized, e.g. 3×3 or 4×4 depending on how
many datasets exist).

**Inputs:** `project` (dict of folder paths, as returned by `create_project_structure`)

**Output:** One thumbnail PNG per dataset, plus one combined grid PNG,
`output/maps/project_outputs_overview.png`.

**Returns:** Path to the combined overview PNG (str)


In [ ]:
def visualize_project_outputs(project: dict) -> str:
    """Scan the project folders and build a contact-sheet overview of every output.

    Parameters
    ----------
    project : dict
        Folder-path dictionary as returned by ``create_project_structure``.

    Returns
    -------
    str
        Path to the combined contact-sheet PNG.
    """
    shp_sources = ["overlay", "status", "boundary", "roads", "facilities"]
    tif_sources = ["flood"]
    csv_sources = ["if", "accessibility", "behaviour", "ba", "master", "tri"]

    shp_files, tif_files, csv_files = [], [], []
    for key in shp_sources:
        shp_files += sorted(Path(project[key]).glob("*.shp"))
    for key in tif_sources:
        tif_files += sorted(Path(project[key]).glob("*.tif"))
    for key in csv_sources:
        csv_files += sorted(Path(project[key]).glob("*.csv"))

    maps_dir = project["maps"]
    thumbs = []
    for shp in shp_files:
        thumbs.append(visualize_vector(str(shp), str(maps_dir / f"{shp.stem}_thumb.png"), shp.stem))
    for tif in tif_files:
        thumbs.append(visualize_raster(str(tif), str(maps_dir / f"{tif.stem}_thumb.png"), tif.stem))
    for csvf in csv_files:
        thumbs.append(visualize_csv(str(csvf), str(maps_dir / f"{csvf.stem}_thumb.png"), csvf.stem))

    n = len(thumbs)
    ncols = math.ceil(math.sqrt(n))
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        if i < n:
            img = mpimg.imread(thumbs[i])
            ax.imshow(img)
            ax.set_title(Path(thumbs[i]).stem.replace("_thumb", ""), fontsize=8)
        ax.axis("off")

    overview_path = str(maps_dir / "project_outputs_overview.png")
    fig.tight_layout()
    fig.savefig(overview_path, dpi=110)
    plt.close(fig)
    print(f"Contact sheet with {n} panels saved to: {overview_path}")
    return overview_path


overview_path = visualize_project_outputs(PROJECT)

## `run_pipeline()` — full automatic execution

**Purpose:** Execute the entire historical TRI pipeline — dummy data generation through TRI and
visualization — in one call, with no manual intervention.

**Inputs:** `base_dir` (str, project root; a fresh directory is safe to pass here)

**Output:** Every intermediate/output file described in this notebook, plus a printed
end-of-run summary table.

**Returns:** `dict` with paths to every key output plus the final TRI DataFrame

**Processing steps:** exactly the pipeline described in the project specification —
Generate Dummy Data → Road-Flood Overlay → Road Status → IF → Accessibility → Travel
Behaviour → BA → Master Dataset → TRI → Visualize All Outputs → Print Summary.


In [ ]:
def run_pipeline(base_dir: str = "/content/project_run") -> dict:
    """Execute the full historical Transport Resilience Index pipeline end-to-end.

    Parameters
    ----------
    base_dir : str
        Root directory for this pipeline run. Created fresh if it does not exist.

    Returns
    -------
    dict
        Paths to every major output file, plus the final TRI DataFrame under key "tri_df".
    """
    print("=" * 70)
    print("RUNNING FULL TRI PIPELINE")
    print("=" * 70)

    proj = create_project_structure(base_dir)

    print("\n[1/10] Generating dummy data...")
    b_path, b_gdf = generate_boundary(str(proj["boundary"] / "boundary.shp"))
    r_path, r_gdf = generate_roads(str(proj["roads"] / "roads.shp"))
    f_path = generate_flood_raster(str(proj["flood"] / "flood_depth.tif"))
    fac_path, fac_gdf = generate_facilities(str(proj["facilities"] / "facilities.shp"))
    t_path = generate_travel_survey(str(proj["travel"] / "travel_survey.csv"), r_gdf, b_gdf)

    print("\n[2/10] Road-Flood Overlay...")
    ov_path = road_flood_overlay(r_path, f_path, str(proj["overlay"] / "road_overlay.shp"))

    print("\n[3/10] Road Status Classification...")
    st_path = classify_road_status(ov_path, str(proj["status"] / "road_status.shp"))

    print("\n[4/10] Infrastructure Functionality (IF)...")
    if_p = calculate_if(st_path, b_path, str(proj["if"] / "if.csv"))

    print("\n[5/10] Accessibility Index (AI)...")
    ai_p = calculate_accessibility(st_path, fac_path, b_path, str(proj["accessibility"] / "accessibility.csv"))

    print("\n[6/10] Travel Behaviour...")
    beh_p = calculate_travel_behaviour(t_path, str(proj["behaviour"] / "behaviour.csv"))

    print("\n[7/10] Behavioural Adaptation (BA)...")
    ba_p = calculate_behavioural_adaptation(beh_p, str(proj["ba"] / "ba.csv"))

    print("\n[8/10] Master Dataset...")
    master_p = build_master_dataset(if_p, ai_p, ba_p, str(proj["master"] / "master_resilience.csv"))

    print("\n[9/10] Transport Resilience Index (TRI)...")
    tri_p = calculate_tri(master_p, str(proj["tri"] / "tri.csv"))
    final_tri = pd.read_csv(tri_p)

    print("\n[10/10] Visualizing all outputs...")
    overview_p = visualize_project_outputs(proj)

    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE")
    print("=" * 70)
    print(f"\nFinal Transport Resilience Index by ward (from {tri_p}):\n")
    print(final_tri.sort_values("TRI", ascending=False).to_string(index=False))
    print(f"\nMost resilient ward : {final_tri.loc[final_tri['TRI'].idxmax(), 'ward_id']} "
          f"(TRI={final_tri['TRI'].max():.2f})")
    print(f"Least resilient ward: {final_tri.loc[final_tri['TRI'].idxmin(), 'ward_id']} "
          f"(TRI={final_tri['TRI'].min():.2f})")
    print(f"\nAll outputs available under: {proj['base']}")
    print(f"Combined visual overview    : {overview_p}")

    return {
        "project": proj,
        "boundary": b_path, "roads": r_path, "flood": f_path,
        "facilities": fac_path, "travel": t_path,
        "overlay": ov_path, "status": st_path,
        "if": if_p, "accessibility": ai_p,
        "behaviour": beh_p, "ba": ba_p,
        "master": master_p, "tri": tri_p,
        "overview": overview_p, "tri_df": final_tri,
    }

## Final execution

This single cell runs the entire notebook's logic from a completely fresh project directory —
proving the pipeline is reproducible and requires no manual steps.


In [ ]:
results = run_pipeline("/content/project_final_run")

## Notebook complete

The historical **Transport Resilience Index** pipeline is fully implemented, documented, and
reproducible:

`Infrastructure Functionality (IF)` → `Accessibility Index (AI)` → `Behavioural Adaptation (BA)`
→ `Transport Resilience Index (TRI)`

Every dataset — synthetic inputs, every intermediate result, every final index, and every
visualization — is written to disk under the project's `input/`, `intermediate/`, and `output/`
folders, and every function is independently re-runnable from those files. No machine learning,
prediction, or generative-AI component was used anywhere in this pipeline: every score reflects
only what the (synthetic, but spatially consistent) historical data shows.
